In [0]:
# Databricks environment configuration
catalog_name = "learning_projects"
schema_name = "retail_project"
volume_name = "source_files"

source_base_path = (
    f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"
)

bronze_table_prefix = f"{catalog_name}.{schema_name}"
print(source_base_path)
print(bronze_table_prefix)

In [0]:
from pyspark.sql.types import(StructType, StructField, IntegerType, StringType)

customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True)
])

from pyspark.sql.functions import current_timestamp, col

bronze_customer_df = (
    spark.read
    .option("header", True)
    .schema(customer_schema)
    .csv(f"{source_base_path}/olist_customers_dataset.csv")
    .select(
        "*",
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("_ingestion_timestamp", current_timestamp())
)
bronze_customer_df.show(5, truncate = False)


In [0]:
bronze_customer_df.printSchema()
bronze_customer_df.show(5, truncate = False)
print(f"customer row :", bronze_customer_df.count())

In [0]:
bronze_customer_df.write\
.format("delta")\
.mode("overwrite")\
.saveAsTable(f"{bronze_table_prefix}.bronze_customers")

In [0]:

spark.sql(f"""
    SELECT COUNT(*) AS customer_count
    FROM {bronze_table_prefix}.bronze_customers
""").show()

In [0]:
spark.sql(f"""
    DESCRIBE DETAIL {bronze_table_prefix}.bronze_customers
""").show(truncate=False)

In [0]:
from pyspark.sql.types import(StructType, StructField, IntegerType, StringType, TimestampType)

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", TimestampType(), True),
    StructField("order_approved_at", TimestampType(), True),
    StructField("order_delivered_carrier_Date", TimestampType(), True ),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True) 
])
# "order_id","customer_id","order_status","order_purchase_timestamp","order_approved_at","order_delivered_carrier_date","order_delivered_customer_date","order_estimated_delivery_date"

from pyspark.sql.functions import current_timestamp, col

bronze_orders_df = (
    spark.read
    .option("header", True)
    .schema(orders_schema)
    .csv(f"{source_base_path}/olist_orders_dataset.csv")
    .select(
        "*",
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("_ingestion_timestamp", current_timestamp())
)


In [0]:
bronze_orders_df.printSchema()
bronze_orders_df.show(5, truncate = False)
print(f"orders row :", bronze_orders_df.count())

In [0]:
(
    bronze_orders_df.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{bronze_table_prefix}.bronze_orders")
)

In [0]:
spark.sql(f"""
    SELECT COUNT(*) AS customer_count
    FROM {bronze_table_prefix}.bronze_orders
""").show()

In [0]:
from pyspark.sql.types import (StructField, StructType, StringType, IntegerType, DoubleType)

#"order_id","order_item_id","product_id","seller_id","shipping_limit_date","price","freight_value"
order_items_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", TimestampType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True)
])

from pyspark.sql.functions import current_timestamp, col

bronze_order_items_df = (
    spark.read
    .option("header", True)
    .schema(order_items_schema)
    .csv(f"{source_base_path}/olist_order_items_dataset.csv")
    .select(
        "*",
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("_ingestion_timestamp", current_timestamp())
)

In [0]:
(
    bronze_order_items_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{bronze_table_prefix}.bronze_order_items")
)

In [0]:
spark.sql(f"""
    SELECT COUNT(*) AS customer_count
    FROM {bronze_table_prefix}.bronze_order_items
""").show()

In [0]:

from pyspark.sql.types import (StructField, StructType, StringType, IntegerType)
products_schema= StructType([
    StructField("product_id", StringType(), True),
    StructField("product_category_name", StringType(), True),
    StructField("product_name_lenght", IntegerType(), True),
    StructField("product_description_lenght", IntegerType(), True),
    StructField("product_photos_qty", IntegerType(), True),
    StructField("product_weight_g", IntegerType(), True),
    StructField("product_length_cm", IntegerType(), True),
    StructField("product_height_cm", IntegerType(), True),
    StructField("product_width_cm", IntegerType(), True)
])

from pyspark.sql.functions import current_timestamp, col
products_df = (
    spark.read
    .option("header", True)
    .schema(products_schema)
    .csv(f"{source_base_path}/olist_products_dataset.csv")
    .select("*",
            col("_metadata.file_path").alias("_source_file")
            )
    .withColumn("_ingestion_timestamp", current_timestamp())
)



In [0]:
(
    products_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(f"{bronze_table_prefix}.bronze_products")
)

In [0]:
spark.sql(f"""
    SELECT COUNT(*) AS customer_count
    FROM {bronze_table_prefix}.bronze_products
""").show()